# Q1a — Data Preprocessing

**W&B run:** _filled in after first run_

This notebook fetches 20 Newsgroups, runs the project's preprocessing pipeline, and produces the two descriptive plots referenced in the report:

- class balance bar chart
- document-length histogram (in tokens, post-preprocessing)


In [ ]:
%load_ext autoreload
%autoreload 2

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from nlp_project import set_seed
from nlp_project.data import load_20ng, preprocess

set_seed()
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)


In [ ]:
# Load + preprocess
train_docs, train_labels, test_docs, test_labels, label_names = load_20ng(remove=True)
print(f"train docs: {len(train_docs)}, test docs: {len(test_docs)}")
print(f"first label: {label_names[train_labels[0]]}")

train_tokens = preprocess(train_docs, drop_stopwords=True)
print(f"first doc, first 20 tokens: {train_tokens[0][:20]}")


In [ ]:
# Class balance plot
counts = Counter(train_labels.tolist())
order = sorted(counts.keys())
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([label_names[i] for i in order], [counts[i] for i in order])
ax.set_ylabel("number of training docs")
ax.set_title("20 Newsgroups — class balance (train split)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "class_balance.png", dpi=150)
plt.show()


In [ ]:
# Doc-length histogram
lengths = np.array([len(t) for t in train_tokens])
print(f"median: {int(np.median(lengths))}, mean: {lengths.mean():.1f}, "
      f"p95: {int(np.percentile(lengths, 95))}, max: {lengths.max()}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.clip(lengths, 0, 1000), bins=50)
ax.set_xlabel("document length (tokens, clipped at 1000)")
ax.set_ylabel("count")
ax.set_title("Document length after preprocessing")
fig.tight_layout()
fig.savefig(FIG_DIR / "doc_length_hist.png", dpi=150)
plt.show()


In [ ]:
# Vocab size and most frequent tokens
vocab = Counter()
for toks in train_tokens:
    vocab.update(toks)
print(f"vocab size: {len(vocab)}")
print(f"top 10: {vocab.most_common(10)}")


## Notes for the report

- Class balance is mild (≈400–600 docs/class) — accuracy is roughly comparable to macro-F1.
- Median doc length after preprocessing is ~80 tokens; p95 ~400. The MLP sees fixed-size doc vectors so length is a sanity check, not a hyperparameter.
- Vocab size and most-frequent tokens go in the report's data section.
